# 02. Build the Map

This notebook reads the small files made by `01datapreparation.ipynb` and draws the web app: one map showing how busy each Rapid Rail station is, when people use it, and (Version 2) what surrounds it, (Version 3) who lives around it, (Version 4) how its use has changed since 2023, and (Version 5) how it changes on public holidays and festive seasons. The surroundings come from `03context1.ipynb` and the population and income from `04population1.ipynb`, the monthly figures from `05timeline1.ipynb`, and the holiday effects from `06holidays1.ipynb`.

| Section | What it does |
|---|---|
| 0 | **Visual settings: the place to experiment** (shapes, colours, sizes, lines, text) |
| 1 | Load the prepared files |
| 2 | Base map and boundaries |
| 3 | Station markers |
| 4 | Click-to-show destinations |
| 5 | Title, legend and notes |
| 6 | Save the web app |

## 0. Visual settings

Colours can be any CSS colour: a name (`"red"`, `"teal"`) or a hex code (`"#1f77b4"`). A colour picker such as [htmlcolorcodes.com](https://htmlcolorcodes.com) gives hex codes.

In [302]:
# ---------------- Station markers ----------------
MARKER_SHAPE     = "circle"    # <- "circle", "square" or "diamond"
MARKER_MIN_SIZE  = 4           # <- width in pixels of the quietest station
MARKER_MAX_SIZE  = 48          # <- width in pixels of the busiest station (Bukit Bintang)
MARKER_OPACITY   = 1        # <- 0 = invisible, 1 = solid
MARKER_BORDER_COLOUR = "#333333" # <- outline around each marker
MARKER_BORDER_WIDTH  = 1.5       # <- outline thickness in pixels (0 = no outline)

# ---------------- Marker colours: weekend ratio ----------------
# weekend_ratio = weekend and holiday entries / working-day entries
COLOUR_COMMUTER = "#E0F2FE"    # <- stations that empty at weekends (ratio around 0.3)
COLOUR_MIXED    = "#30BCF8"    # <- stations in between (ratio around 0.75)
COLOUR_WEEKEND  = "#0C2363"    # <- stations busier at weekends (ratio 1.2 or more)
RATIO_COMMUTER  = 0.3          # <- ratio at which COLOUR_COMMUTER is reached
RATIO_MIXED     = 0.75         # <- ratio shown in COLOUR_MIXED
RATIO_WEEKEND   = 1.2          # <- ratio at which COLOUR_WEEKEND is reached

# ---------------- Destination lines (shown when a station is clicked) ----------------
LINE_COLOUR     = "#fbff00"    # <- colour of the lines to the top destinations
LINE_MIN_WIDTH  = 4            # <- thinnest line in pixels
LINE_MAX_WIDTH  = 12           # <- thickest line in pixels (the busiest destination overall)
LINE_OPACITY    = 0.9          # <- 0 = invisible, 1 = solid
LINE_DASHED     = False        # <- True for dashed lines
HIGHLIGHT_COLOUR = "#fbff00"   # <- ring drawn around the clicked station

# ---------------- Boundaries ----------------
DISTRICT_LINE_COLOUR = "#444444"   # <- district outlines
DISTRICT_LINE_WIDTH  = 2           # <- thickness in pixels
SHOW_DISTRICTS_AT_START = False  # <- True to show district outlines when the page opens
CONSTITUENCY_LINE_COLOUR = "#000000"  # <- constituency outlines
CONSTITUENCY_LINE_WIDTH  = 1          # <- thickness in pixels
SHOW_CONSTITUENCIES_AT_START = False  # <- True to show constituency outlines when the page opens

# ---------------- Station surroundings ----------------
SHOW_CONTEXT_IN_TOOLTIP = True     # <- True to list nearby shops, food and offices when hovering over a station
SHOW_LANDUSE_AT_START   = True    # <- True to show the land-use layer when the page opens
LANDUSE_OPACITY         = 0.45     # <- fill strength of the land-use layer (0 to 1)
LANDUSE_COLOURS = {                # <- fill colour for each land-use group
    "residential": "#ffe343",
    "commercial":  "#EE7DF8",
    "industrial":  "#5E462E",
    "green":       "#0fca28",
}
LANDUSE_LABELS = {                 # <- text shown in the legend for each land-use group
    "residential": "Residential",
    "commercial":  "Commercial and retail",
    "industrial":  "Industrial",
    "green":       "Parks and green space",
}

# ---------------- Station types and income ----------------
COLOUR_BY_AT_START = "weekend_ratio"   # <- how stations are coloured when the page opens: "weekend_ratio", "station_type", "growth" or "festive"
TYPE_COLOURS = {                       # <- colour for each station type (used when "Station type" is chosen on the map)
    "Home-end commuter":   "#1f78b4",  # <- blue
    "Work-end commuter":   "#6a3d9a",  # <- purple
    "Local neighbourhood": "#33a02c",  # <- green
    "Leisure destination": "#e31a1c",  # <- red
}
SHOW_INCOME_AT_START = False           # <- True to show the household income layer when the page opens
INCOME_COLOURS  = ["#fff7ec", "#b30000"]  # <- [lowest income, highest income] colours for the district shading
INCOME_OPACITY  = 0.45                 # <- fill strength of the income layer (0 to 1)
NOTES_OPEN_AT_START = False            # <- True to show the "About the data" panel opened when the page opens

# ---------------- Base map and text ----------------
BASEMAP     = "OpenStreetMap"  # <- background map; "OpenStreetMap" needs no key
START_ZOOM  = 12               # <- higher = closer in; 11 shows the whole Klang Valley
MAP_CENTRE  = [3.08, 101.65]   # <- [latitude, longitude] the map opens on
FONT        = "Segoe UI, Helvetica, Arial, sans-serif"  # <- font for the title and panels
TITLE       = "MALAYSIA (KLANG VALLEY): RAPID RAIL" # <- main heading
SUBTITLE    = ("Klang Valley Rapid Rail network entries by station:<br>"
               "• Level of activity<br>"
               "• Surrounding area (land-use; 500m radius)<br>"
               "• Surrounding population (station type)<br>"
               "• Changes over time (timeline)<br>"
               "• Public holidays & feastive seasons")
PANEL_BACKGROUND = "rgba(255, 255, 255, 0.92)"  # <- background of the title, legend and info panels
FT_PREFIX   = ""               # <- shown instead of "W.P. " for Kuala Lumpur and Putrajaya, e.g. "" or "Federal Territory of "
TOOLTIP_FONT_SIZE = 14         # <- text size in pixels for all hover tooltips (Leaflet's default is about 12)

# ---------------- Changes over time ----------------
GROWTH_COLOURS = ["#FFFFFF", "#F7A1B0", "#DC143C"]   # <- [fell, no change, grew strongly] colours for "Change since 2023"
GROWTH_STOPS   = [-20, 0, 100]                    # <- % change at which each colour above is reached (beyond the ends = end colour)
COLOUR_NEW     = "#ff9900"                        # <- stations that opened after 2023, so cannot be compared
TIMELINE_SPEED_MS = 500                           # <- milliseconds per month when "Play" is pressed (lower = faster)
TIMELINE_HIDE_EMPTY = True                        # <- True hides a station in months before it opened
TIMELINE_NOTE = ("Station size follows the chosen month; colours stay as set above.<br>"
                 "Slide fully to the right for the {period}.")   # <- small text under the time slider; {period} becomes e.g. "Jun to Aug 2026 average"

# ---------------- Title panel buttons ----------------
BUTTON_LABELS = {                        # <- text on the colour buttons in the title panel
    "ratio":  "WEEKEND RATIO",
    "type":   "STATION TYPE",
    "growth": "CHANGES SINCE 2023",
    "festive": "FESTIVE EFFECT",
}

# ---------------- Public holidays and festive seasons ----------------
FESTIVE_COLOURS = ["#542788", "#f7f7f7", "#e66101"]  # <- [much quieter, same as a normal week, busier] colours for "Festive effect"
FESTIVE_STOPS   = [-50, 0, 30]                        # <- % effect at which each colour above is reached (beyond the ends = end colour)
FESTIVE_NO_DATA = "#bbbbbb"                           # <- stations with no data for the chosen festival (opened later)
FESTIVE_LEAVE_OUT = ["Other public holidays"]         # <- groups from 06holidays1 not offered on the map
FESTIVE_AT_START = "Hari Raya Aidilfitri"             # <- festival selected when the page opens

## 1. Load the prepared files

In [303]:
from pathlib import Path
import json, math
import pandas as pd
import geopandas as gpd
import folium
from branca.colormap import LinearColormap
from branca.element import MacroElement, Template, Element

PROCESSED = Path("data/processed")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

stations = gpd.read_file(PROCESSED / "stations.geojson")
flows = pd.read_csv(PROCESSED / "flows_top.csv")
districts = gpd.read_file(PROCESSED / "districts.geojson")
constituencies = gpd.read_file(PROCESSED / "constituencies.geojson")
meta = json.loads((PROCESSED / "metadata.json").read_text())

# Version 2: surroundings from 03context1.ipynb
context = pd.read_csv(PROCESSED / "station_context.csv")
ctx_meta = json.loads((PROCESSED / "context_metadata.json").read_text())
landuse = gpd.read_file(PROCESSED / "landuse_near_stations.geojson")
stations = stations.merge(context, on="key", how="left")

# Version 3: residents, station types and income from 04population1.ipynb
population = pd.read_csv(PROCESSED / "station_population.csv")
pop_meta = json.loads((PROCESSED / "population_metadata.json").read_text())
districts_income = gpd.read_file(PROCESSED / "districts_income.geojson")
stations = stations.merge(population, on="key", how="left")

# Version 4: monthly entries and change since 2023 from 05timeline1.ipynb
monthly = pd.read_csv(PROCESSED / "station_monthly.csv")
growth = pd.read_csv(PROCESSED / "station_growth.csv")
tl_meta = json.loads((PROCESSED / "timeline_metadata.json").read_text())
stations = stations.merge(growth, on="key", how="left")
SEASON_LABEL = " to ".join(pd.Timestamp(f"2000-{d}").strftime("%b") for d in tl_meta["season"])   # e.g. "Jun to Aug"

stations = stations.sort_values("avg_daily_entries", ascending=False).reset_index(drop=True)
print(f"{len(stations)} stations, {len(flows)} flows, period {meta['period_start']} to {meta['period_end']}")
print(f"Surroundings within {ctx_meta['radius_m']} m, OpenStreetMap data fetched {ctx_meta['osm_fetched_on']}; "
      f"{stations['total_places'].notna().sum()} stations matched")
print(f"Residents within {pop_meta['radius_m']} m and station types for {stations['station_type'].notna().sum()} stations")

def show_name(d):
    """District name for display: 'W.P. Kuala Lumpur' -> FT_PREFIX + 'Kuala Lumpur'."""
    return str(d).replace("W.P. ", FT_PREFIX)
print(f"Monthly entries for {monthly['key'].nunique()} stations, {tl_meta['months'][0]} to {tl_meta['months'][-1]}; "
      f"{(stations['status'] == 'compared').sum()} stations compared {SEASON_LABEL} {tl_meta['compare_from']} with {tl_meta['compare_to']}")

# Version 5: public holidays and festive seasons from 06holidays1.ipynb
holiday = pd.read_csv(PROCESSED / "station_holidays.csv")
hol_meta = json.loads((PROCESSED / "holiday_metadata.json").read_text())
FEST_LIST = [f for f in hol_meta["festivals"] if f not in FESTIVE_LEAVE_OUT]
fest_table = holiday.pivot(index="key", columns="festival", values="effect_pct").reindex(columns=FEST_LIST)
print(f"Festive effects for {fest_table.notna().any(axis=1).sum()} stations: {', '.join(FEST_LIST)}")

154 stations, 770 flows, period 2026-06-01 to 2026-08-31
Surroundings within 500 m, OpenStreetMap data fetched 2026-09-30; 154 stations matched
Residents within 500 m and station types for 154 stations
Monthly entries for 154 stations, 2023-01 to 2026-09; 135 stations compared Jun to Aug 2023 with 2026
Festive effects for 154 stations: Hari Raya Aidilfitri, Chinese New Year, Deepavali, Hari Raya Haji, Christmas and New Year, National Day (Merdeka)


## 2. Base map and boundaries

In [304]:
# VISUAL: centre, zoom, background
m = folium.Map(location=MAP_CENTRE, zoom_start=START_ZOOM, tiles=BASEMAP, control_scale=True, prefer_canvas=False)
# VISUAL: grey, faded background (remove this line for full colour)
m.get_root().header.add_child(Element("<style>.leaflet-tile-pane { filter: grayscale(70%); opacity: 0.8; }</style>"))
# VISUAL: keeps the layer menu in front of the destinations panel when it opens
m.get_root().header.add_child(Element("<style>.leaflet-top.leaflet-right { z-index: 1100; }</style>"))
# VISUAL: hover tooltip text size
m.get_root().header.add_child(Element(f"<style>.leaflet-tooltip {{ font-size: {TOOLTIP_FONT_SIZE}px; font-family: {FONT}; line-height: 1.4; }}</style>"))

# VISUAL: district outline colour
# VISUAL: district outline width
districts["district_label"] = districts["district"].map(show_name)   # VISUAL: display names
folium.GeoJson(
    districts, name="Districts", show=SHOW_DISTRICTS_AT_START,
    style_function=lambda f: {"color": DISTRICT_LINE_COLOUR, "weight": DISTRICT_LINE_WIDTH, "fillOpacity": 0},
    tooltip=folium.GeoJsonTooltip(["district"], labels=False),
).add_to(m)

# VISUAL: constituency outline colour
# VISUAL: constituency outline width
# VISUAL: "4 4" = dash pattern
folium.GeoJson(
    constituencies, name="Parliamentary constituencies", show=SHOW_CONSTITUENCIES_AT_START,
    style_function=lambda f: {"color": CONSTITUENCY_LINE_COLOUR, "weight": CONSTITUENCY_LINE_WIDTH, "dashArray": "4 4", "fillOpacity": 0},           
    tooltip=folium.GeoJsonTooltip(["parlimen"], labels=False),
).add_to(m)

# Land-use areas within the station circles (switch on from the layer menu)
landuse["label"] = landuse["land_use"].map(LANDUSE_LABELS).fillna(landuse["land_use"])   # VISUAL: hover text uses LANDUSE_LABELS
folium.GeoJson(
    landuse, name=f"Land use within {ctx_meta['radius_m']} m of stations", show=SHOW_LANDUSE_AT_START,
    style_function=lambda f: {"fillColor": LANDUSE_COLOURS.get(f["properties"]["land_use"], "#cccccc"),   # VISUAL: land-use colours
                              "fillOpacity": LANDUSE_OPACITY, "color": LANDUSE_COLOURS.get(f["properties"]["land_use"], "#cccccc"),
                              "weight": 0.5},
    tooltip=folium.GeoJsonTooltip(["label"], labels=False),
).add_to(m)

# Version 3: median household income by district (switch on from the layer menu)
income_scale = LinearColormap(INCOME_COLOURS, vmin=districts_income["income_median"].min(),
                              vmax=districts_income["income_median"].max())                     # VISUAL: income colours
districts_income["income_label"] = districts_income.apply(
    lambda r: f"{show_name(r['district'])}: median RM {r['income_median']:,.0f} a month ({r['income_year']:.0f})", axis=1)
folium.GeoJson(
    districts_income, name="Median household income by district", show=SHOW_INCOME_AT_START,
    style_function=lambda f: {"fillColor": income_scale(f["properties"]["income_median"]), "fillOpacity": INCOME_OPACITY, "color": "#555555", "weight": 1},
    tooltip=folium.GeoJsonTooltip(["income_label"], labels=False),
).add_to(m)

## 3. Station markers

Marker **area** grows with average daily entries (the width grows with the square root), so a station with four times the entries looks four times as big, not sixteen. Colour shows the weekend ratio. Busier stations are drawn underneath quieter ones so small stations stay visible.

In [305]:
# VISUAL: the three colours
colour_scale = LinearColormap(
    colors=[COLOUR_COMMUTER, COLOUR_MIXED, COLOUR_WEEKEND],
    index=[RATIO_COMMUTER, RATIO_MIXED, RATIO_WEEKEND],
    vmin=RATIO_COMMUTER, vmax=RATIO_WEEKEND)

max_entries = stations["avg_daily_entries"].max()

def marker_size(entries):
    """Width in pixels: area proportional to entries."""
    return MARKER_MIN_SIZE + (MARKER_MAX_SIZE - MARKER_MIN_SIZE) * math.sqrt(entries / max_entries)

def marker_colour(ratio):
    return colour_scale(min(max(ratio, RATIO_COMMUTER), RATIO_WEEKEND))   # values beyond the ends use the end colour

def marker_html(size, colour, attrs=""):
    """The marker is a small box drawn with CSS; its shape comes from MARKER_SHAPE."""
    shape_css = {
        "circle":  "border-radius: 50%;",                  # VISUAL: round corners all the way = circle
        "square":  "border-radius: 2px;",                  # VISUAL: slightly rounded square corners
        "diamond": "border-radius: 2px; transform: rotate(45deg) scale(0.8);",  # VISUAL: square turned 45 degrees
    }[MARKER_SHAPE]
    return (f'<div {attrs} style="width:{size:.0f}px; height:{size:.0f}px; background:{colour}; '
            f'opacity:{MARKER_OPACITY}; border:{MARKER_BORDER_WIDTH}px solid {MARKER_BORDER_COLOUR}; '
            f'box-sizing:border-box; box-shadow:0 0 2px rgba(0,0,0,0.5); cursor:pointer; {shape_css}"></div>')

def service_html(s):
    """Note for stations with a shorter record, e.g. the Shah Alam Line (LRT3) opened in June 2026."""
    note = s.get("service_note", "")
    return f"<span style='color:#b35900'>New: {note}</span><br>" if isinstance(note, str) and note else ""

def tooltip_html(s):
    return (f"<b>{s['name']}</b> <span style='color:#777'>({s['codes']})</span><br>"
            f"{s['avg_daily_entries']:,.0f} entries a day<br>"
            + service_html(s)
            + f"Working days: {s['avg_working_day']:,.0f} &middot; Weekends and holidays: {s['avg_weekend_holiday']:,.0f}<br>"
            f"Weekend ratio: {s['weekend_ratio']:.2f}<br>"
            f"<span style='color:#777'>{show_name(s['district'])}</span><br>"
            + (context_html(s) if SHOW_CONTEXT_IN_TOOLTIP else "")
            + population_html(s)
            + growth_html(s)
            + f"<i style='color:#777'>Click for top destinations and surroundings</i>")

def population_html(s):
    """Residents within walking distance and the station type."""
    if pd.isna(s.get("residents")):
        return ""
    epr = "" if pd.isna(s["entries_per_resident"]) else f"<br>{s['entries_per_resident']:.2f} daily entries into the station per resident"
    return (f"<span style='color:#555'>About {s['residents']:,.0f} residents within {pop_meta['radius_m']} m{epr}<br>"
            f"Station type: <b>{s['station_type']}</b></span><br>")

# VISUAL: colours for "Change since 2023" (GROWTH_COLOURS and GROWTH_STOPS in Section 0)
growth_scale = LinearColormap(colors=GROWTH_COLOURS, index=GROWTH_STOPS, vmin=GROWTH_STOPS[0], vmax=GROWTH_STOPS[-1])

def growth_colour(s):
    if s.get("status") == "compared":
        return growth_scale(min(max(s["change_pct"], GROWTH_STOPS[0]), GROWTH_STOPS[-1]))
    return COLOUR_NEW if s.get("status") == "new since 2023" else "#999999"

def growth_html(s):
    """Change in average daily entries between the same season of two years."""
    y0, y1 = tl_meta["compare_from"], tl_meta["compare_to"]
    if s.get("status") == "compared":
        return (f"Change since {y0}: <b>{s['change_pct']:+.0f}%</b> "
                f"<span style='color:#777'>({SEASON_LABEL}: {s['avg_2023']:,.0f} to {s['avg_2026']:,.0f} a day)</span><br>")
    if s.get("status") == "new since 2023":
        return f"<span style='color:#777'>Opened after {y0}: no change to compare</span><br>"
    return ""

# VISUAL: colours for "Festive effect" (FESTIVE_COLOURS and FESTIVE_STOPS in Section 0)
festive_scale = LinearColormap(colors=FESTIVE_COLOURS, index=FESTIVE_STOPS, vmin=FESTIVE_STOPS[0], vmax=FESTIVE_STOPS[-1])

def festive_attrs(s):
    """One colour per festival, stored on the marker as data-c-f0, data-c-f1, ..."""
    row = fest_table.loc[s["key"]] if s["key"] in fest_table.index else pd.Series(index=FEST_LIST, dtype=float)
    cols = [FESTIVE_NO_DATA if pd.isna(v) else festive_scale(min(max(v, FESTIVE_STOPS[0]), FESTIVE_STOPS[-1])) for v in row]
    return " ".join(f'data-c-f{i}="{c}"' for i, c in enumerate(cols))

def type_colour(t):
    return TYPE_COLOURS.get(t, "#999999")

def context_html(s):
    """A short line about what is within walking distance."""
    if pd.isna(s.get("total_places")):
        return ""
    return (f"<span style='color:#555'>Within {ctx_meta['radius_m']} m: {s['shops']:.0f} shops, "
            f"{s['food']:.0f} food and drink, {s['offices']:.0f} offices<br>"
            f"Main land use: {s['main_land_use']}</span><br>")

station_layer = folium.FeatureGroup(name="Stations").add_to(m)
marker_names = {}   # station key -> the marker's name in the web page, used in Section 4

for rank, s in stations.iterrows():
    size = marker_size(s["avg_daily_entries"])
    mk = folium.Marker(
        location=[s["lat"], s["lon"]],
        icon=folium.DivIcon(html=marker_html(size, marker_colour(s["weekend_ratio"]),
                                             attrs=f'class="stn" data-c-ratio="{marker_colour(s["weekend_ratio"])}" '
                                                   f'data-c-type="{type_colour(s["station_type"])}" '
                                                   f'data-c-growth="{growth_colour(s)}" ' + festive_attrs(s)),
                            icon_size=(size, size), icon_anchor=(size / 2, size / 2)),
        tooltip=folium.Tooltip(tooltip_html(s), sticky=True),
        z_index_offset=rank * 10,          # busier stations first = underneath
    ).add_to(station_layer)
    marker_names[s["key"]] = mk.get_name()

print(f"Drew {len(marker_names)} markers as {MARKER_SHAPE}s, {MARKER_MIN_SIZE} to {MARKER_MAX_SIZE} px")

Drew 154 markers as circles, 4 to 48 px


## 4. Click-to-show destinations

Clicking a station draws lines to its top destinations and lists them in a panel. Clicking the map background clears them. Folium cannot do this on its own, so this cell adds a short piece of JavaScript. **You do not need to edit it**: its colours and widths come from Section 0.

In [306]:
# Data handed to the web page
CTX_COLS = ["shops", "food", "offices", "education", "healthcare", "tourism_leisure", "total_places"]
station_info = {s["key"]: {"name": s["name"], "lat": s["lat"], "lon": s["lon"], "entries": s["avg_daily_entries"], "note": s["service_note"] if isinstance(s["service_note"], str) else "", "size": marker_size(s["avg_daily_entries"]),
                           "ctx": None if pd.isna(s["total_places"]) else {**{c: int(s[c]) for c in CTX_COLS}, "main": s["main_land_use"]},
                           "pop": None if pd.isna(s["residents"]) else {
                               "residents": float(s["residents"]), "type": s["station_type"],
                               "epr": None if pd.isna(s["entries_per_resident"]) else float(s["entries_per_resident"]),
                               "income": None if pd.isna(s["income_median"]) else float(s["income_median"]),
                               "year": None if pd.isna(s["income_year"]) else int(s["income_year"]), "district": show_name(s["district"])}}
                for _, s in stations.iterrows()}
# Version 4: change since 2023 and monthly entries (one value per month, None before a station opened)
for key, info in station_info.items():
    g = stations.loc[stations["key"] == key].iloc[0]
    info["growth"] = None if pd.isna(g["status"]) else {
        "status": g["status"], "pct": None if pd.isna(g["change_pct"]) else float(g["change_pct"]),
        "a": None if pd.isna(g["avg_2023"]) else float(g["avg_2023"]), "b": None if pd.isna(g["avg_2026"]) else float(g["avg_2026"]),
        "note": g["note"] if isinstance(g["note"], str) else ""}
# Version 5: festive effect per festival (None where the station has no data)
for key, info in station_info.items():
    row = fest_table.loc[key] if key in fest_table.index else None
    info["fest"] = [None if row is None or pd.isna(row[f]) else float(row[f]) for f in FEST_LIST]
MONTHS = tl_meta["months"]
month_table = monthly.pivot(index="key", columns="month", values="avg_daily_entries").reindex(columns=MONTHS)
monthly_info = {k: [None if pd.isna(v) else float(v) for v in month_table.loc[k]] if k in month_table.index else [None] * len(MONTHS)
                for k in station_info}

flow_info = {k: g[["dest_key", "avg_daily_trips", "share_of_origin"]].values.tolist() for k, g in flows.sort_values("rank").groupby("origin_key")}
style = {"lineColour": LINE_COLOUR, "lineMin": LINE_MIN_WIDTH, "lineMax": LINE_MAX_WIDTH,
         "lineOpacity": LINE_OPACITY, "dashed": LINE_DASHED, "highlight": HIGHLIGHT_COLOUR,
         "maxTrips": float(flows["avg_daily_trips"].max()), "topN": int(meta["top_n"]), "radius": int(ctx_meta["radius_m"]),
         "season": SEASON_LABEL, "festivals": FEST_LIST,
         "festQuiet": FESTIVE_COLOURS[0], "festBusy": FESTIVE_COLOURS[-1], "y0": tl_meta["compare_from"], "y1": tl_meta["compare_to"]}

CLICK_JS = r"""
(function() {
  var map = {{ this._parent.get_name() }};
  var stations = __STATIONS__;
  var flows = __FLOWS__;
  var style = __STYLE__;
  var markers = {__MARKERS__};
  var monthly = __MONTHLY__;
  var months = __MONTHS__;
  var current = null;
  window.klMap = map; window.klStations = stations; window.klMarkers = markers;
  window.klMonthly = monthly; window.klMonths = months;
  window.klRefreshPanel = function() { if (current) { showFlows(current); } };
  var layer = L.layerGroup().addTo(map);
  var panel = document.getElementById("flow-panel");
  var fmt = function(n) { return Math.round(n).toLocaleString("en-GB"); };

  function clearFlows() {
    layer.clearLayers();
    current = null;
    panel.innerHTML = "<i>Click a station to see where its passengers go.</i>";
  }
  function showFlows(key) {
    layer.clearLayers();
    current = key;
    var o = stations[key];
    var rows = "";
    (flows[key] || []).forEach(function(f, i) {
      var d = stations[f[0]];
      if (!d) { return; }
      var w = style.lineMin + (style.lineMax - style.lineMin) * Math.sqrt(f[1] / style.maxTrips);
      L.polyline([[o.lat, o.lon], [d.lat, d.lon]], {
        color: style.lineColour, weight: w, opacity: style.lineOpacity,
        dashArray: style.dashed ? "8 6" : null, interactive: false
      }).addTo(layer);
      rows += "<tr><td>" + (i + 1) + ".</td><td>" + d.name + "</td><td style='text-align:right'>" + fmt(f[1]) +
              "</td><td style='text-align:right;color:#777'>" + Math.round(f[2] * 100) + "%</td></tr>";
    });
    L.circleMarker([o.lat, o.lon], {radius: o.size / 2 + 5, color: style.highlight, weight: 3,
                                    fill: false, interactive: false}).addTo(layer);
    panel.innerHTML = "<b>" + o.name + "</b>: top " + style.topN + " destinations<br>" +
      "<span style='color:#777'>" + fmt(o.entries) + " entries a day</span>" +
      (o.note ? "<br><span style='color:#b35900'>New: " + o.note + "</span>" : "") +
      "<table style='margin-top:6px;border-collapse:collapse'><tr style='color:#777'><td></td><td>Destination</td>" +
      "<td style='text-align:right;padding-left:8px'>Trips/day</td><td style='text-align:right;padding-left:8px'>Share</td></tr>" +
      rows + "</table>" + contextBlock(o.ctx) + populationBlock(o.pop) + growthBlock(o.growth, key) + festiveBlock(o.fest);
  }
  function populationBlock(p) {
    if (!p) { return ""; }
    var epr = (p.epr === null) ? "few residents nearby" : p.epr.toFixed(2) + " daily entries into the station per resident";
    var inc = (p.income === null) ? "" : "<br><span style='color:#777'>" + p.district + " median household income:<br>RM " +
              fmt(p.income) + " a month (" + p.year + ")</span>";
    return "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>Who uses it</b><br>" +
      "About " + fmt(p.residents) + " residents within " + style.radius + " m<br>" + epr + "<br>" +
      "Station type: <b>" + p.type + "</b>" + inc + "</div>";
  }
  function growthBlock(g, key) {
    if (!g) { return ""; }
    var head = "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>Change since " + style.y0 + "</b><br>";
    var body;
    if (g.status === "compared") {
      var col = g.pct < 0 ? "#b2182b" : "#DC143C";
      body = style.season + " " + style.y0 + ": " + fmt(g.a) + " a day<br>" + style.season + " " + style.y1 + ": " + fmt(g.b) + " a day " +
             "<b style='color:" + col + "'>(" + (g.pct >= 0 ? "+" : "") + g.pct.toFixed(0) + "%)</b>" +
             (g.note ? "<br><span style='color:#777'>" + g.note + "</span>" : "");
    } else {
      body = "<span style='color:#777'>Opened after " + style.y0 + ", so there is no earlier period to compare.</span>";
    }
    return head + body + sparkline(monthly[key]) + "</div>";
  }
  function festiveBlock(f) {
    if (!f || f.every(function(v) { return v === null; })) { return ""; }
    var rows = "";
    style.festivals.forEach(function(name, i) {
      var v = f[i];
      if (v === null) { return; }
      var col = v < 0 ? style.festQuiet : style.festBusy;
      rows += "<tr><td style='padding:1px 14px 1px 0'>" + name + "</td><td style='text-align:right;color:" + col + "'><b>" +
              (v >= 0 ? "+" : "") + v.toFixed(0) + "%</b></td></tr>";
    });
    return "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>Festive seasons</b> " +
      "<span style='color:#777'>(against a normal week)</span><table style='margin-top:4px;border-collapse:collapse'>" + rows + "</table></div>";
  }
  function sparkline(vals) {
    // Small chart of monthly average daily entries; the grey line marks the month chosen on the time slider
    var W = 290, H = 70, pad = 14, n = vals.length;
    var max = Math.max.apply(null, vals.filter(function(v) { return v !== null; }).concat([1]));
    var x = function(i) { return pad + (W - 2 * pad) * i / (n - 1); };
    var y = function(v) { return H - 14 - (H - 22) * v / max; };
    var path = "", pen = false;
    vals.forEach(function(v, i) {
      if (v === null) { pen = false; return; }
      path += (pen ? "L" : "M") + x(i).toFixed(1) + "," + y(v).toFixed(1); pen = true;
    });
    var ticks = "";
    months.forEach(function(mo, i) {
      if (mo.slice(5) === "01") {
        ticks += "<line x1='" + x(i) + "' x2='" + x(i) + "' y1='" + (H - 14) + "' y2='" + (H - 10) + "' stroke='#999'/>" +
                 "<text x='" + x(i) + "' y='" + (H - 1) + "' font-size='10' fill='#777' text-anchor='middle'>" + mo.slice(0, 4) + "</text>";
      }
    });
    var sel = (window.klMonthIndex !== undefined && window.klMonthIndex < n) ?
      "<line x1='" + x(window.klMonthIndex) + "' x2='" + x(window.klMonthIndex) + "' y1='2' y2='" + (H - 14) + "' stroke='#888' stroke-dasharray='3 2'/>" : "";
    return "<div style='margin-top:6px;color:#777;font-size:12px'>Entries a day by month (peak " + fmt(max) + ")</div>" +
      "<svg width='" + W + "' height='" + H + "' style='display:block'>" +
      "<line x1='" + pad + "' x2='" + (W - pad) + "' y1='" + (H - 14) + "' y2='" + (H - 14) + "' stroke='#ccc'/>" +
      ticks + sel + "<path d='" + path + "' fill='none' stroke='#333' stroke-width='1.5'/></svg>";
  }
  function contextBlock(c) {
    if (!c) { return ""; }
    var cell = function(label, n) {
      return "<td style='padding:1px 10px 1px 0'>" + label + "</td><td style='text-align:right;padding-right:14px'><b>" + fmt(n) + "</b></td>";
    };
    return "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>Within " + style.radius + " m</b>" +
      "<table style='margin-top:4px;border-collapse:collapse'>" +
      "<tr>" + cell("Shops", c.shops) + cell("Offices", c.offices) + "</tr>" +
      "<tr>" + cell("Food and drink", c.food) + cell("Education", c.education) + "</tr>" +
      "<tr>" + cell("Tourism and leisure", c.tourism_leisure) + cell("Healthcare", c.healthcare) + "</tr>" +
      "</table><span style='color:#777'>Main land use: " + c.main + " &middot; " + fmt(c.total_places) + " places mapped</span></div>";
  }
  Object.keys(markers).forEach(function(key) {
    markers[key].on("click", function(e) { L.DomEvent.stopPropagation(e); showFlows(key); });
  });
  map.on("click", clearFlows);
  clearFlows();
})();
"""

js = (CLICK_JS.replace("__STATIONS__", json.dumps(station_info))
              .replace("__FLOWS__", json.dumps(flow_info))
              .replace("__STYLE__", json.dumps(style))
              .replace("__MONTHLY__", json.dumps(monthly_info))
              .replace("__MONTHS__", json.dumps(MONTHS))
              .replace("__MARKERS__", ", ".join(f'"{k}": {v}' for k, v in marker_names.items())))
click_js = MacroElement()
click_js._template = Template("{% macro script(this, kwargs) %}" + js + "{% endmacro %}")
m.add_child(click_js)
print(f"Click behaviour added for {len(flow_info)} stations")

Click behaviour added for 154 stations


## 5. Title, legend and notes

In [307]:
panel_css = (f"position:absolute; z-index:1000; background:{PANEL_BACKGROUND}; font-family:{FONT}; "   # VISUAL: panel look
             "padding:10px 14px; border-radius:6px; box-shadow:0 1px 5px rgba(0,0,0,0.3); font-size:14px;")

period = f"{pd.Timestamp(meta['period_start']):%d %b %Y} to {pd.Timestamp(meta['period_end']):%d %b %Y}"

# Colour switch inside the title panel
btn = ("border:1px solid #888; background:#fff; border-radius:4px; padding:2px 8px; margin-left:4px; "
       "font-family:inherit; font-size:12px; cursor:pointer;")
fest_start = FEST_LIST.index(FESTIVE_AT_START) if FESTIVE_AT_START in FEST_LIST else 0
fest_options = "".join(f'<option value="{i}"{" selected" if i == fest_start else ""}>{f}</option>' for i, f in enumerate(FEST_LIST))

switch_html = (f'<div style="margin-top:6px; font-size:13px;">'
               f'<div>Colour stations by:</div><div style="margin-top:4px; margin-left:-4px;">'   # VISUAL: label on its own line, buttons in rows below it
               f'<div>'                                                           # first row of buttons
               f'<button id="btn-ratio" style="{btn}" onclick="setStationColour(\'ratio\')">{BUTTON_LABELS["ratio"]}</button>'
               f'<button id="btn-type" style="{btn}" onclick="setStationColour(\'type\')">{BUTTON_LABELS["type"]}</button>'
               f'<button id="btn-growth" style="{btn}" onclick="setStationColour(\'growth\')">{BUTTON_LABELS["growth"]}</button></div>'
               f'<div style="margin-top:4px;">'                                   # second row, lined up under the first button
               f'<button id="btn-festive" style="{btn}" onclick="setStationColour(\'festive\')">{BUTTON_LABELS["festive"]}</button>'
               f'<select id="fest-select" onchange="setFestival(this.value)" style="{btn} display:none;">{fest_options}</select></div>'
               f'</div></div>')
SWITCH_JS = """
<script>
var FEST = __FEST__;
window.klFestive = __FEST_START__;
function setFestival(i) {
  window.klFestive = +i;
  document.getElementById('fest-name').textContent = FEST[i].name;
  document.getElementById('fest-net').textContent = FEST[i].net;
  if (window.klColourMode === 'festive') { setStationColour('festive'); }
}
function setStationColour(mode) {
  window.klColourMode = mode;
  var attr = {ratio: 'cRatio', type: 'cType', growth: 'cGrowth', festive: 'cF' + window.klFestive}[mode];
  document.querySelectorAll('.stn').forEach(function(el) { el.style.background = el.dataset[attr]; });
  ['ratio', 'type', 'growth', 'festive'].forEach(function(k) {
    document.getElementById('legend-' + k).style.display = (k === mode) ? 'block' : 'none';
    document.getElementById('btn-' + k).style.fontWeight = (k === mode) ? 'bold' : 'normal';
  });
  document.getElementById('fest-select').style.display = (mode === 'festive') ? 'inline-block' : 'none';
}
window.addEventListener('load', function() { setFestival(window.klFestive); setStationColour('__START__'); });
</script>
""".replace("__START__", {"station_type": "type", "growth": "growth", "festive": "festive"}.get(COLOUR_BY_AT_START, "ratio")
).replace("__FEST_START__", str(fest_start)).replace("__FEST__", json.dumps([
    {"name": f, "net": f"Whole network: {hol_meta['network_effect_pct'][f]:+.1f}%."} for f in FEST_LIST]))

# Title panel, top left
title_html = f"""
<div style="{panel_css} top:14px; left:60px; max-width:600px;">
  <div style="font-size:20px; font-weight:600;">{TITLE}</div>
  <div style="color:#555; margin-top:2px;">{SUBTITLE}</div>
  <div style="color:#777; margin-top:4px; font-size:12px;">Average per day, {period}</div>
  {switch_html}
</div>"""

# Destinations panel, top right (below the layer switcher)
flow_panel_html = (f'<div id="flow-panel" style="{panel_css} top:60px; right:12px; width:320px; '
                   f'max-height:calc(100vh - 140px); overflow-y:auto; z-index:1001;"></div>')   # VISUAL: scrolls when long

# Legend, bottom left: size examples and colour bar
examples = [1000, 10000, int(round(max_entries, -3))]
size_rows = "".join(
    f'<div style="display:flex; align-items:center; margin:3px 0;">'
    f'<div style="width:{MARKER_MAX_SIZE + 4}px; display:flex; justify-content:center;">{marker_html(marker_size(n), "#bbbbbb")}</div>'
    f'<span style="margin-left:6px;">{n:,} entries a day</span></div>' for n in examples)
landuse_keys = "".join(
    f'<span><span style="display:inline-block; width:10px; height:10px; background:{c}; opacity:{max(LANDUSE_OPACITY, 0.6)}; '
    f'margin-right:4px; vertical-align:middle;"></span>{LANDUSE_LABELS.get(g, g)}</span>' for g, c in LANDUSE_COLOURS.items())
type_keys = "".join(
    f'<div style="display:flex; align-items:center; margin:2px 0;"><span style="display:inline-block; width:12px; height:12px; '
    f'background:{c}; border:1px solid {MARKER_BORDER_COLOUR}; margin-right:6px;"></span>{t}</div>' for t, c in TYPE_COLOURS.items())


gradient = f"linear-gradient(to right, {COLOUR_COMMUTER}, {COLOUR_MIXED}, {COLOUR_WEEKEND})"
# VISUAL: gradient for "Change since 2023", with each colour placed at its GROWTH_STOPS position
# VISUAL: gradient for "Festive effect", with each colour placed at its FESTIVE_STOPS position
festive_gradient = "linear-gradient(to right, " + ", ".join(
    f"{c} {100 * (v - FESTIVE_STOPS[0]) / (FESTIVE_STOPS[-1] - FESTIVE_STOPS[0]):.0f}%" for c, v in zip(FESTIVE_COLOURS, FESTIVE_STOPS)) + ")"
growth_gradient = "linear-gradient(to right, " + ", ".join(
    f"{c} {100 * (v - GROWTH_STOPS[0]) / (GROWTH_STOPS[-1] - GROWTH_STOPS[0]):.0f}%" for c, v in zip(GROWTH_COLOURS, GROWTH_STOPS)) + ")"
legend_html = f"""
<div style="{panel_css} bottom:50px; left:14px; width:300px;">
  <div style="font-weight:600; margin-bottom:4px;">Station size: level of activity</div>
  {size_rows}
  <div id="legend-ratio">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: weekend ratio</div>
  <div style="height:12px; border-radius:3px; background:{gradient}; border:1px solid #ccc;"></div>
  <div style="display:flex; justify-content:space-between; color:#555; font-size:12px; margin-top:2px;">
    <span>Commuter<br>{RATIO_COMMUTER}</span><span style="text-align:center">Mixed<br>{RATIO_MIXED}</span><span style="text-align:right">Weekend<br>{RATIO_WEEKEND}+</span>
  </div>
  <div style="color:#777; font-size:12px; margin-top:6px;">Ratio = weekend and public holiday entries divided by working-day entries.</div>
  </div>
  <div id="legend-type" style="display:none;">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: station type</div>
  {type_keys}
  <div style="color:#777; font-size:12px; margin-top:6px;">Weekday or weekend: weekend ratio below or above {pop_meta['ratio_split']:.2f}.
  Residents or visitors: entries per resident below or above {pop_meta['entries_per_resident_split']:.2f}.</div>
  </div>
  <div id="legend-growth" style="display:none;">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: change since {tl_meta['compare_from']}</div>
  <div style="height:12px; border-radius:3px; background:{growth_gradient}; border:1px solid #ccc;"></div>
  <div style="display:flex; justify-content:space-between; color:#555; font-size:12px; margin-top:2px;">
    <span>{GROWTH_STOPS[0]:+d}% or less</span><span>{GROWTH_STOPS[1]:+d}%</span><span style="text-align:right">{GROWTH_STOPS[-1]:+d}% or more</span>
  </div>
  <div style="display:flex; align-items:center; margin-top:4px; font-size:12px;"><span style="display:inline-block; width:12px; height:12px;
  background:{COLOUR_NEW}; border:1px solid {MARKER_BORDER_COLOUR}; margin-right:6px;"></span>Opened after {tl_meta['compare_from']} (not compared)</div>
  <div style="color:#777; font-size:12px; margin-top:6px;">Average daily entries, {SEASON_LABEL} {tl_meta['compare_from']} against {SEASON_LABEL} {tl_meta['compare_to']}.
  Whole network: {tl_meta['network_change_pct']:+.1f}%.</div>
  </div>
  <div id="legend-festive" style="display:none;">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: festive effect, <span id="fest-name"></span></div>
  <div style="height:12px; border-radius:3px; background:{festive_gradient}; border:1px solid #ccc;"></div>
  <div style="display:flex; justify-content:space-between; color:#555; font-size:12px; margin-top:2px;">
    <span>{FESTIVE_STOPS[0]:+d}% or less</span><span>{FESTIVE_STOPS[1]:+d}% (normal)</span><span style="text-align:right">{FESTIVE_STOPS[-1]:+d}% or more</span>
  </div>
  <div style="display:flex; align-items:center; margin-top:4px; font-size:12px;"><span style="display:inline-block; width:12px; height:12px;
  background:{FESTIVE_NO_DATA}; border:1px solid {MARKER_BORDER_COLOUR}; margin-right:6px;"></span>No data (opened after that festival)</div>
  <div style="color:#777; font-size:12px; margin-top:6px;">Entries during the festival against a normal week at the same station, 2023 to 2026.
  <span id="fest-net"></span></div>
  </div>
  <div style="font-weight:600; margin:8px 0 4px;">Land-use layer <span style="font-weight:400; color:#777; font-size:12px;">(switch on/off in the layer menu)</span></div>
  <div style="display:flex; flex-wrap:wrap; gap:4px 12px; font-size:12px;">{landuse_keys}</div>
</div>"""

# Notes, bottom right: data note and sources
notes_html = f"""
<div style="{panel_css} bottom:30px; right:12px; width:300px; font-size:12px; color:#555;">
<details {"open" if NOTES_OPEN_AT_START else ""}>
  <summary style="cursor:pointer;"><b>About the data</b> <span style="color:#999;">(click to open or close)</span></summary>
  Level of activity = average daily entries (tap-ins). Stations that opened during the period are averaged over the days they have data: the Shah Alam Line (LRT3) opened on 29 June 2026 with free rides until 31 July, so its figures cover August only. Exits at 32 Putrajaya Line stations are missing in the source data, so activity uses entries only and no destination line ends at those stations.<br><br>
  Surroundings: places and land use within {ctx_meta['radius_m']} m, from OpenStreetMap (fetched {ctx_meta['osm_fetched_on']}). Counts show what volunteers have mapped, not a census.<br><br>
  Station types compare entries with residents within {pop_meta['radius_m']} m (Kontur population, 2023). Stations with park-and-ride or feeder buses draw users from further away and can look like destinations.<br><br>
  Time slider and change since {tl_meta['compare_from']}: average daily entries by calendar month from {pd.Timestamp(MONTHS[0]):%b %Y} to {pd.Timestamp(MONTHS[-1]):%b %Y}{" (the last month may be incomplete)" if pd.Timestamp(MONTHS[-1]) >= pd.Timestamp(meta['period_end']).to_period('M').to_timestamp() else ""}. Change compares {SEASON_LABEL} {tl_meta['compare_from']} with {SEASON_LABEL} {tl_meta['compare_to']}; interchanges are compared only on the platforms open in both years.<br><br>
  Festive effect: entries during each festival (for example the Hari Raya week, from the day before to six days after) against the same station's usual entries on the same day of the week in the four weeks before and after, pooled over 2023 to 2026.<br><br>
  Sources: data.gov.my<br>
  (Prasarana ridership and GTFS, CC BY 4.0); DOSM boundaries and household income; Kontur population (CC BY 4.0); map &copy; OpenStreetMap contributors.<br>
  Last updated: {meta['prepared_on']}.
</details>
</div>"""

# Time slider, bottom centre
network_by_month = month_table.sum(min_count=1).reindex(MONTHS)
period_label = f"{pd.Timestamp(meta['period_start']):%b} to {pd.Timestamp(meta['period_end']):%b %Y} average"
timeline_html = f"""
<div id="timeline" style="{panel_css} bottom:30px; left:50%; transform:translateX(-50%); width:400px;">
  <div style="display:flex; align-items:center; gap:8px;">
    <button id="tl-play" style="{btn} margin-left:0; width:58px;">&#9654; Play</button>
    <input id="tl-slider" type="range" min="0" max="{len(MONTHS)}" value="{len(MONTHS)}" step="1" style="flex:1; accent-color:#555;">
  </div>
  <div style="margin-top:4px;"><b id="tl-label"></b> <span id="tl-net" style="color:#777; font-size:12px;"></span></div>
  <div style="color:#777; font-size:12px;">{TIMELINE_NOTE.format(period=period_label)}</div>
</div>"""   # VISUAL: slider panel position and width

TIMELINE_JS = """
<script>
window.addEventListener('load', function() {
  var cfg = __CFG__;
  var months = window.klMonths, monthly = window.klMonthly, stations = window.klStations, markers = window.klMarkers;
  var n = months.length, slider = document.getElementById('tl-slider'), playBtn = document.getElementById('tl-play');
  var timer = null;
  var fmt = function(v) { return Math.round(v).toLocaleString('en-GB'); };
  var monthName = function(mo) { return new Date(mo + '-15').toLocaleString('en-GB', {month: 'short', year: 'numeric'}); };
  var size = function(v) { return cfg.minSize + (cfg.maxSize - cfg.minSize) * Math.sqrt(v / cfg.maxEntries); };

  function resize(key, v) {
    var mk = markers[key], icon = mk._icon, o = mk.options.icon.options;
    var hide = (v === null || v <= 0) && cfg.hideEmpty;
    var s = size((v === null || v <= 0) ? 0 : v);
    // Keep Leaflet's own copy up to date, so the size survives the layer being switched off and on
    o.iconSize = [s, s]; o.iconAnchor = [s / 2, s / 2];
    o.html = o.html.replace(/width:[0-9.]+px; height:[0-9.]+px/, 'width:' + s.toFixed(1) + 'px; height:' + s.toFixed(1) + 'px');
    stations[key].size = s;
    if (!icon) { return; }
    icon.style.display = hide ? 'none' : '';
    icon.style.width = s + 'px'; icon.style.height = s + 'px';
    icon.style.marginLeft = (-s / 2) + 'px'; icon.style.marginTop = (-s / 2) + 'px';
    var box = icon.querySelector('.stn');
    if (box) { box.style.width = s + 'px'; box.style.height = s + 'px'; }
  }
  function show(i) {
    window.klMonthIndex = i;
    Object.keys(markers).forEach(function(key) {
      resize(key, i < n ? monthly[key][i] : stations[key].entries);
    });
    document.getElementById('tl-label').textContent = i < n ? monthName(months[i]) : cfg.periodLabel;
    document.getElementById('tl-net').textContent = 'Network: ' + fmt(i < n ? cfg.network[i] : cfg.periodNetwork) + ' entries a day';
    if (window.klRefreshPanel) { window.klRefreshPanel(); }
  }
  function stop() { clearInterval(timer); timer = null; playBtn.innerHTML = '&#9654; Play'; }
  playBtn.addEventListener('click', function() {
    if (timer) { stop(); return; }
    if (+slider.value >= n - 1) { slider.value = 0; show(0); }
    playBtn.innerHTML = '&#10074;&#10074; Pause';
    timer = setInterval(function() {
      var i = +slider.value + 1;
      if (i >= n) { stop(); return; }
      slider.value = i; show(i);
    }, cfg.speed);
  });
  slider.addEventListener('input', function() { stop(); show(+slider.value); });
  // Leaflet redraws markers when the Stations layer is switched back on: reapply colours and sizes
  window.klMap.on('overlayadd', function() { setStationColour(window.klColourMode || 'ratio'); show(+slider.value); });
  show(+slider.value);
});
</script>
""".replace("__CFG__", json.dumps({
    "minSize": MARKER_MIN_SIZE, "maxSize": MARKER_MAX_SIZE, "maxEntries": float(max_entries),
    "speed": TIMELINE_SPEED_MS, "hideEmpty": TIMELINE_HIDE_EMPTY, "periodLabel": period_label,
    "network": [None if pd.isna(v) else float(v) for v in network_by_month],
    "periodNetwork": float(stations["avg_daily_entries"].sum())}))

for html in (title_html, flow_panel_html, legend_html, notes_html, timeline_html, SWITCH_JS, TIMELINE_JS):
    m.get_root().html.add_child(Element(html))

folium.LayerControl(collapsed=True).add_to(m)

## 6. Save the web app

In [308]:
out_file = OUTPUT / "kl_mobility_v5.html"
m.get_root().header.add_child(Element("<title>MALAYSIA(KLANG VALLEY): RAPID RAIL RIDES</title>"))
m.save(out_file)
print(f"Saved {out_file} ({out_file.stat().st_size / 1e3:.0f} KB). Open it in a browser for the full-size view.")

# Also save as docs/index.html, the page GitHub Pages publishes (Version 1 is kept in backup/v1)
import shutil
Path("docs").mkdir(exist_ok=True)
shutil.copyfile(out_file, Path("docs") / "index.html")   # a copy, not a second save (saving twice repeats parts of the page)
print("Also saved docs/index.html, ready to upload to GitHub")
m

Saved output\kl_mobility_v5.html (1133 KB). Open it in a browser for the full-size view.
Also saved docs/index.html, ready to upload to GitHub


.venv\Scripts\python -m http.server 8000 --directory docs

http://localhost:8000/
___
**END**